# 01 · Build data

Reads the CSVs from `00_espn_pull` and the Almost Points data from FantasyLuck, then works out every number the site shows: the weekly recap, every matchup, standings, a page for every team, Draft & Moves, and the league history. Everything is saved to one file, `data/site_data.pkl`, which `02_build_site` turns into pages.

Nothing here is specific to one manager. Every team and every matchup gets the same treatment.

In [ ]:
from pathlib import Path
from collections import defaultdict
import json, math, re, statistics
import numpy as np
import pandas as pd

LEAGUE_ID = 35924907
THROUGH_WEEK = None        # None builds the latest week where every game has a score

DATA = Path("data")
LUCK_DIR = Path("../FantasyLuck/data")      # FantasyLuck saves site_<season>.json here
OUT_FILE = DATA / "site_data.pkl"
P = DATA / f"espn_{LEAGUE_ID}_"

## Load

The settings, teams, scores, lineups, drafts, and transactions for every season.

In [ ]:
SET = dict(pd.read_csv(f"{P}settings.csv").values)
TM = pd.read_csv(f"{P}teams.csv")
SCH = pd.read_csv(f"{P}schedule.csv")
LU = pd.read_csv(f"{P}lineups.csv")
DR = pd.read_csv(f"{P}draft.csv")
TX = pd.read_csv(f"{P}transactions.csv")

LEAGUE_NAME = str(SET["name"])
PLAYOFF_TEAMS = int(SET["playoff_teams"])
REG_WEEKS = int(SET["regular_season_weeks"])
SEASON = int(SCH.season.max())
REC_PTS = float(SET.get("score_REC", 0))
SCORING = "PPR" if REC_PTS >= 1 else ("Half PPR" if REC_PTS >= 0.5 else "Standard")

# The latest week where every game has a score for both teams
cur_games = SCH[(SCH.season == SEASON) & (SCH.bracket == "NONE")]
scored = cur_games.groupby("week").apply(lambda g: ((g.home_score > 0) & (g.away_score > 0)).all())
WEEK = int(THROUGH_WEEK or scored[scored].index.max())
print(LEAGUE_NAME, "·", SEASON, "· through Week", WEEK)

## Lineup slots

The starting slots and how many of each. The best possible lineup fills them in this order: one-position slots first, then the FLEX from whoever's left.

In [ ]:
SLOTS = {"QB": 1, "RB": 2, "WR": 2, "TE": 1, "FLEX": 1, "K": 1, "DEF": 1}
STARTER_SLOTS = set(SLOTS)

def optimal_df(df):
    # A slot left empty scores 0, so a player below 0 is never part of the best lineup
    pool = df[(df.slot != "IR") & (df.points > 0)].sort_values("points", ascending=False)
    used, total, picks = set(), 0.0, []
    for pos, n in (("QB", 1), ("K", 1), ("DEF", 1), ("TE", 1), ("RB", 2), ("WR", 2)):
        c = pool[(pool.position == pos) & (~pool.player_id.isin(used))].head(n)
        used |= set(c.player_id); total += c.points.sum(); picks += list(c.player_id)
    c = pool[(pool.position.isin(["RB", "WR", "TE"])) & (~pool.player_id.isin(used))].head(1)
    used |= set(c.player_id); total += c.points.sum(); picks += list(c.player_id)
    return total, set(picks)

## Teams and weekly scores

Teams are tracked by their league slot and labeled with this season's names, so a slot that changed managers keeps one history. `SC` holds each team's score by week and `opp` its opponent.

In [ ]:
cur = TM[TM.season == SEASON]
ID2AB = {r.team_id: str(r.abbrev) for r in cur.itertuples()}
AB2ID = {v: k for k, v in ID2AB.items()}
NAME = {str(r.abbrev): r.team_name.strip() for r in cur.itertuples()}
MGR = {str(r.abbrev): r.manager for r in cur.itertuples()}
TEAMS = [ID2AB[i] for i in sorted(ID2AB)]
N = len(TEAMS)

s_cur = SCH[(SCH.season == SEASON) & (SCH.week <= WEEK)]
SC = {t: [0.0] * WEEK for t in TEAMS}
opp = {t: [None] * WEEK for t in TEAMS}
for r in s_cur.itertuples():
    a, b = ID2AB[r.home_team_id], ID2AB[r.away_team_id]; k = r.week - 1
    SC[a][k] = round(r.home_score, 2); SC[b][k] = round(r.away_score, 2)
    opp[a][k] = b; opp[b][k] = a
SCORES = SC

## Lineups, best lineups, and projections

For every team and week: the best lineup the roster allowed, the points left on the bench, and the projected points of the lineup that was started.

In [ ]:
L26 = LU[(LU.season == SEASON) & (LU.week <= WEEK)].copy()
L26["t"] = L26.team_id.map(ID2AB)

OPT = {t: [0.0] * WEEK for t in TEAMS}
BENCH = {t: [0.0] * WEEK for t in TEAMS}
PROJD = {t: [0.0] * WEEK for t in TEAMS}
for (t, w), g in L26.groupby(["t", "week"]):
    o, _ = optimal_df(g); OPT[t][w - 1] = o
    st = g[g.slot.isin(STARTER_SLOTS)]
    PROJD[t][w - 1] = st.projected.sum()
    BENCH[t][w - 1] = o - st.points.sum()
LINEW = {t: [100 * SC[t][k] / OPT[t][k] if OPT[t][k] else 100 for k in range(WEEK)] for t in TEAMS}
MAXPF = {t: sum(OPT[t]) for t in TEAMS}

## Reversed trades

A commissioner undo shows up as later trades between the same two teams that put every player back. Starting from each trade, this adds the next trades between that pair (within 48 hours) until every player's net move is zero. All of those trades are treated as never having happened.

In [ ]:
def undone_trades(tx, hours=48):
    tr = tx[tx.action.str.startswith("TRADE")].copy(); tr["date"] = pd.to_datetime(tr.date, format="mixed")
    moves = sorted([(mid, g.date.iloc[0], frozenset(g.team_id), g) for mid, g in tr.groupby("move_id")], key=lambda m: m[1])
    out = set()
    for i, (mid, d0, pair, _) in enumerate(moves):
        if mid in out:
            continue
        net = defaultdict(int); group = []
        for mj, dj, pj, gj in moves[i:]:
            if pj != pair or (dj - d0).total_seconds() > hours * 3600:
                continue
            group.append(mj)
            for r in gj.itertuples():
                net[(r.player_id, r.team_id)] += 1 if r.action == "TRADE_RECEIVED" else -1
            if len(group) > 1 and all(v == 0 for v in net.values()):
                out |= set(group); break
    return out

UNDONE = undone_trades(TX[TX.season == SEASON])
print("Reversed trades:", sorted(UNDONE) or "none")

## Points lost to luck

Joins each started player's week to the Almost Points data by name. Only the losses count here (flags, QB fault, drops, replay, and stopped short), so a team's luck is the points its starters lost.

In [ ]:
AD = json.load(open(LUCK_DIR / f"site_{SEASON}.json"))
LOSSC = ["flag", "qb_fault", "drop", "replay", "short"]

def norm(n):
    n = re.sub(r"[^a-z ]", "", str(n).lower().replace("-", " "))
    return " ".join(w for w in n.split() if w not in ("jr", "sr", "ii", "iii", "iv", "v"))

LUCK = {}
for p in AD["players"]:
    for w in p["weeks"]:
        if w["week"] <= WEEK:
            LUCK[(norm(p["name"]), w["week"])] = {c: w["cols"][c][0] for c in AD["columns"]}

L26["nn"] = L26.player_name.map(norm)
L26["lost"] = [sum(max(LUCK.get((r.nn, r.week), {}).get(c, 0), 0) for c in LOSSC) for r in L26.itertuples()]

LUCKW = {t: [0.0] * WEEK for t in TEAMS}
LUCKCOL = {t: defaultdict(float) for t in TEAMS}
for r in L26[L26.slot.isin(STARTER_SLOTS)].itertuples():
    LUCKW[r.t][r.week - 1] += r.lost
    for c in LOSSC:
        LUCKCOL[r.t][c] += max(LUCK.get((r.nn, r.week), {}).get(c, 0), 0)

skill = L26[L26.slot.isin(STARTER_SLOTS) & L26.position.isin(["QB", "RB", "WR", "TE"])]
print(f"Starters matched to Almost Points: {skill.nn.isin({k[0] for k in LUCK}).mean():.1%}")

POSL = ["QB", "RB", "WR", "TE", "K", "DEF"]
POS_SPLIT = {t: [L26[(L26.t == t) & L26.slot.isin(STARTER_SLOTS) & (L26.position == p)].points.sum() for p in POSL] for t in TEAMS}
PTS = L26.drop_duplicates(["player_id", "week"]).groupby("player_id").points.sum()

## Recap

This week's games (winner first), the four headline tiles, the six weekly awards, and the ten best players of the week, bench included.

In [ ]:
L10 = L26[L26.week == WEEK].copy()
L10["wk_rank"] = L10.groupby("position").points.rank(ascending=False, method="min").astype(int)
starters10 = L10[L10.slot.isin(STARTER_SLOTS)]

GAMES = []
for r in SCH[(SCH.season == SEASON) & (SCH.week == WEEK)].itertuples():
    a, b = ID2AB[r.home_team_id], ID2AB[r.away_team_id]; sa, sb = r.home_score, r.away_score
    if sb > sa:
        a, b, sa, sb = b, a, sb, sa
    GAMES.append((a, sa, b, sb))
GAMES.sort(key=lambda g: -g[1])
GAMES10 = GAMES

TOP3 = {t: [(r.player_name, r.position, r.points) for r in starters10[starters10.t == t].sort_values("points", ascending=False).head(3).itertuples()] for t in TEAMS}
SC10D = {t: SC[t][WEEK - 1] for t in TEAMS}
HI10 = max(SC10D, key=SC10D.get); lo = min(SC10D, key=SC10D.get)
oppw = lambda t: opp[t][WEEK - 1]
won10 = lambda t: SC10D[t] > SC10D[oppw(t)]
CLOSE10 = min(GAMES, key=lambda g: g[1] - g[3]); BLOW10 = max(GAMES, key=lambda g: g[1] - g[3])
c_, b_ = CLOSE10, BLOW10
TILES = dict(
    top=(f"{SC10D[HI10]:.1f}", f"[Team {HI10}] {'over' if won10(HI10) else 'lost to'} [Team {oppw(HI10)}]", f"[Team {HI10}]"),
    low=(f"{SC10D[lo]:.1f}", f"[Team {lo}], {'beat' if won10(lo) else 'lost to'} [Team {oppw(lo)}]", f"[Team {lo}]"),
    close=(f"{c_[1]-c_[3]:.1f} points", f"[Team {c_[0]}] {c_[1]:.1f}, [Team {c_[2]}] {c_[3]:.1f}", f"{c_[1]-c_[3]:.1f} pts", f"[Team {c_[0]}] over [Team {c_[2]}]"),
    blow=(f"{b_[1]-b_[3]:.1f} points", f"[Team {b_[0]}] {b_[1]:.1f}, [Team {b_[2]}] {b_[3]:.1f}", f"{b_[1]-b_[3]:.1f} pts", f"[Team {b_[0]}] over [Team {b_[2]}]"))

best = starters10.sort_values("points", ascending=False).iloc[0]
fw = min(GAMES, key=lambda g: g[1]); ML10 = max(GAMES, key=lambda g: g[3])
luck10 = {t: LUCKW[t][WEEK - 1] for t in TEAMS}; bench10 = {t: BENCH[t][WEEK - 1] for t in TEAMS}
lk_lo = min(luck10, key=luck10.get); lk_hi = max(luck10, key=luck10.get); bn = max(bench10, key=bench10.get)
AWARDS = [("Best single game", best.player_name, f"{best.points:.1f} for [Team {best.t}]", True),
          ("Fewest points in a win", f"[Team {fw[0]}]", f"{fw[1]:.1f}, won by {fw[1]-fw[3]:.1f}", True),
          ("Least lost to luck", f"[Team {lk_lo}]", f"{luck10[lk_lo]:.1f} lost to luck", True),
          ("Most points in a loss", f"[Team {ML10[2]}]", f"{ML10[3]:.1f}, lost by {ML10[1]-ML10[3]:.1f}", False),
          ("Points left on the bench", f"[Team {bn}]", f"{bench10[bn]:.1f} on the bench", False),
          ("Unluckiest team", f"[Team {lk_hi}]", f"{luck10[lk_hi]:.1f} lost to luck", False)]
top10 = L10[L10.slot != "IR"].sort_values("points", ascending=False).head(10)
TOP = [(r.player_name, r.position, r.nfl_team, r.points, r.lost, r.t, 1 if r.slot in STARTER_SLOTS else 0) for r in top10.itertuples()]
WEEK_SCORES = list(SC10D.values())
_sc = sorted(WEEK_SCORES); MEDIAN = (_sc[len(_sc) // 2 - 1] + _sc[len(_sc) // 2]) / 2
print("Top score:", TILES["top"][0], "· closest game:", TILES["close"][0])

## Matchups

Every game this week. Each lineup is listed slot by slot (slot, player, position, NFL team, points, projection, points lost to luck, the player's rank at his position that week, and TDs), with the bench below.

`standouts` picks each team's stud and dud (against projection), best and worst start, and highest and lowest scorer. A start only counts as a mistake if a bench player who could legally have taken that slot outscored him; players on a bye or ruled out (both projected at 0) aren't real options, so they're left out.

In [ ]:
ORDER_SLOTS = ["QB", "RB", "RB", "WR", "WR", "TE", "FLEX", "K", "DEF"]
tot = lambda L: sum(r[4] for r in L)
proj = lambda L: sum(r[5] for r in L)
luck = lambda L: sum(r[6] for r in L)

def lineup_tuples(t):
    g = L10[L10.t == t]; out = []; used = set()
    for s in ORDER_SLOTS:
        c = g[(g.slot == s) & (~g.player_id.isin(used))].sort_values("points", ascending=False)
        if not len(c):
            out.append((s, "Empty", s, "", 0.0, 0.0, 0.0, 0, 0)); continue
        r = c.iloc[0]; used.add(r.player_id)
        out.append((s, r.player_name, r.position, r.nfl_team if r.position != "DEF" else "", round(r.points, 2),
                    round(r.projected, 2), round(r.lost, 1), int(r.wk_rank), 0))
    bench = [(r.player_name, r.position, r.nfl_team, round(r.points, 2)) for r in g[g.slot == "BN"].sort_values("points", ascending=False).itertuples()]
    unavailable = set(g[g.projected == 0].player_name)
    return out, bench, unavailable

def swaps(L, B, out_):
    # Every fair one-for-one swap of a starter for a bench player: same position, or any RB, WR, or TE
    # for the FLEX. A bench RB, WR, or TE can replace a starter at another of those positions only through
    # the FLEX, when the FLEX player plays that position and outscored the starter he'd replace.
    flex = next((r for r in L if r[0] == "FLEX"), None); out = []
    for s_ in L:
        if s_[1] in out_:
            continue
        for b in B:
            if b[0] in out_:
                continue
            pos = b[1]
            if s_[0] == pos or (s_[0] == "FLEX" and pos in ("RB", "WR", "TE")):
                out.append((s_, b))
            elif s_[0] in ("RB", "WR", "TE") and pos in ("RB", "WR", "TE") and flex and flex[2] == s_[0] and flex[4] > s_[4]:
                out.append((s_, b))
    return out

def standouts(L, B, out_):
    real = [r for r in L if r[2] not in ("K", "DEF")] or L
    d = lambda r: r[4] - r[5]
    sgn = lambda x: ("+" if x > 0 else "−" if x < 0 else "") + f"{abs(x):.1f}"
    stud = max(real, key=d); dud = min(real, key=d)
    hi = max(L, key=lambda r: r[4]); lo = min(L, key=lambda r: r[4])
    sw = swaps(L, B, out_)
    worst = max(sw, key=lambda t: t[1][3] - t[0][4]) if sw else None
    margins = {}
    for s_, b in sw:
        margins.setdefault(s_[1], (s_, []))[1].append(b)
    best = max(margins.values(), key=lambda t: t[0][4] - max(x[3] for x in t[1])) if margins else None
    out = {"Stud": (stud[1], f"{stud[4]:.1f} on a {stud[5]:.1f} projection, {sgn(d(stud))}"),
           "Dud": (dud[1], f"{dud[4]:.1f} on a {dud[5]:.1f} projection, {sgn(d(dud))}")}
    if best:
        s_, bs = best; top = max(bs, key=lambda x: x[3])
        out["Best start"] = (s_[1], f"{s_[4]:.1f}, {s_[4]-top[3]:.1f} more than {top[0]} on the bench")
    if worst and worst[1][3] > worst[0][4]:
        s_, b = worst
        out["Worst start"] = (s_[1], f"{s_[4]:.1f} at {s_[0]}, {b[3]-s_[4]:.1f} fewer than {b[0]} on the bench")
    else:
        out["Worst start"] = ("None", "No bench player would have done better")
    out["Highest scorer"] = (hi[1], f"{hi[4]:.1f}")
    out["Lowest scorer"] = (lo[1], f"{lo[4]:.1f}")
    return out, worst

def story(LC, LD, BD, WD):
    # The matchup's story and its one "if" line, written from the numbers ([Team C] won, [Team D] lost)
    pc, pd_ = proj(LC), proj(LD); lc, ld = luck(LC), luck(LD)
    s1 = (f"[Team C] was projected to {'win' if pc > pd_ else 'lose'} by {abs(pc-pd_):.1f} and won by {tot(LC)-tot(LD):.1f}"
          + (f", beating its projection by {tot(LC)-pc:.1f}." if tot(LC) > pc else f", even {pc-tot(LC):.1f} short of its projection."))
    lk = lambda L: sorted([r for r in L if r[6] > 0.05], key=lambda r: -r[6])[:2]
    led = lambda L: (", led by " + " and ".join(f"{r[1]} ({r[6]:.1f})" for r in lk(L))) if lk(L) else ""
    s2 = f"[Team D]'s starters lost {ld:.1f} points to luck{led(LD)}, against {lc:.1f} for [Team C]."
    topc = max(LC, key=lambda r: r[4]); bb = max(BD, key=lambda b: b[3]) if BD else None
    s3 = f"{topc[1]} carried [Team C] with {topc[4]:.1f}" + (f", while {bb[0]}'s {bb[3]:.1f} sat on [Team D]'s bench." if bb and bb[3] > min(r[4] for r in LD) else ".")
    ws, wb = WD if WD else (None, None)
    if ws and wb and wb[3] > ws[4]:
        m = tot(LD) - ws[4] + wb[3] - tot(LC)
        ifl = (f"[Team D] had started {wb[0]} ({wb[3]:.1f}, bench) at {ws[0]} instead of {ws[1]} ({ws[4]:.1f}), "
               + (f"[Team D] would have won by {m:.1f}." if m > 0 else f"[Team D] would still have lost, by {-m:.1f}."))
    else:
        ifl = "[Team D] had no bench player who would have outscored a starter he could legally replace."
    return " ".join([s1, s2, s3]), ifl

MATCHUPS = []
for a, sa, b, sb in GAMES:
    LC, BC, outc = lineup_tuples(a); LD, BD, outd = lineup_tuples(b)
    STAND_C, WC = standouts(LC, BC, outc); STAND_D, WD = standouts(LD, BD, outd)
    OPT_C, OPT_D = OPT[a][WEEK - 1], OPT[b][WEEK - 1]
    STORY, IFLINE = story(LC, LD, BD, WD)
    MATCHUPS.append(dict(MATCH_A=a, MATCH_B=b, LC=LC, BC=BC, LD=LD, BD=BD, SC_=STAND_C, SD=STAND_D, WC=WC, WD=WD,
                         OPT_C=OPT_C, OPT_D=OPT_D, EFF_C=100 * tot(LC) / OPT_C, EFF_D=100 * tot(LD) / OPT_D,
                         STORY=STORY, IFLINE=IFLINE, SEASON_AVG={"C": sum(SC[a]) / WEEK, "D": sum(SC[b]) / WEEK}))
print(len(MATCHUPS), "matchups")

## Standings

Record, points for and against, lineup score, all-play, record against the weekly median, expected wins (the all-play win rate over the games played), and schedule luck (real wins minus expected wins). Ties in the standings go to points for. `SWAP` is every team's record with every other team's schedule.

In [ ]:
W = WEEK

def record_with_schedule(t, sched_of):
    w = l = 0
    for k in range(W):
        o = opp[sched_of][k]
        if o == t:
            o = sched_of
        if SC[t][k] > SC[o][k]: w += 1
        else: l += 1
    return w, l

rows = []
for t in TEAMS:
    w, l = record_with_schedule(t, t)
    ap_w = sum(1 for k in range(W) for o in TEAMS if o != t and SC[t][k] > SC[o][k]); ap_l = W * (N - 1) - ap_w
    mw = sum(1 for k in range(W) if SC[t][k] > statistics.median(SC[x][k] for x in TEAMS))
    pf = sum(SC[t]); pa = sum(SC[opp[t][k]][k] for k in range(W))
    exp = ap_w / (ap_w + ap_l) * W
    rows.append(dict(t=t, w=w, l=l, pf=pf, pa=pa, ap_w=ap_w, ap_l=ap_l, mw=mw, ml=W - mw, exp=exp, luck=w - exp, lineup=100 * pf / MAXPF[t]))
rows.sort(key=lambda r: (-r["w"], -r["pf"]))
LEFT = REG_WEEKS - W
for i, r in enumerate(rows, 1):
    r["rank"] = i
    others_max = sorted([x["w"] + LEFT for x in rows if x is not r], reverse=True)
    others_cur = sorted([x["w"] for x in rows if x is not r], reverse=True)
    cut = PLAYOFF_TEAMS - 1
    r["status"] = "Clinched" if r["w"] > others_max[cut] else ("Eliminated" if r["w"] + LEFT < others_cur[cut] else "")
SWAP = {t: {s: record_with_schedule(t, s) for s in TEAMS} for t in TEAMS}
pd.DataFrame(rows)[["rank", "t", "w", "l", "pf", "ap_w", "ap_l", "exp", "luck", "lineup"]].round(1)

## Teams

One set of numbers per team, built by `team_data`. A few notes on the choices behind them:

- **Who held the team down** compares each starter with enough starts (at least 40% of the weeks, and never fewer than 3) against the league's average starter at his position.
- **Injuries** count a player who started the week before he got hurt, or who was a regular starter (started at least half his healthy weeks with this team, and at least twice). He's out when he's on the roster but projected and scored 0 outside his bye, or after he was dropped while hurt (on IR now and hasn't scored for anyone since). Each stint runs until he scores again for anyone. Every week he's out, his points per start before the injury are compared with the lowest-scoring starter at his position (or the FLEX for RB, WR, and TE), the spot he'd have taken back. Bye weeks are skipped.
- **Overturned touchdowns** are plays for this team's starters that cost a touchdown through a flag, replay, or being stopped short, worth 5.5 points or more.

In [ ]:
st_all = L26[L26.slot.isin(STARTER_SLOTS)]
POS_AVG = {p: st_all[st_all.position == p].points.mean() for p in ("QB", "RB", "WR", "TE")}
MIN_STARTS = max(3, math.ceil(WEEK * 0.4))
_z = L26[L26.position != "DEF"].assign(zero=L26.projected == 0).groupby(["nfl_team", "week"]).zero.mean().reset_index()
BYEWK = {t: set(g[g.zero >= 0.8].week) for t, g in _z.groupby("nfl_team")}
scored_wk = L26[L26.points != 0].groupby("player_id").week.apply(set).to_dict()
on_ir_now = set(L26.sort_values("week").groupby("player_id").injury.last().loc[lambda x: x == "INJURY_RESERVE"].index)
n_started = {t: L26[(L26.t == t) & L26.slot.isin(STARTER_SLOTS)].player_id.nunique() for t in TEAMS}
ordn = lambda n: f"{n}{'th' if 10 <= n % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(n % 10, 'th')}"
healthy = lambda r: not (r.projected == 0 and r.points == 0)

def injuries_for(T0, allme):
    INJ, seen = [], set()
    for k in range(2, WEEK + 1):
        for r in allme[(allme.week == k - 1) & (allme.slot != "IR")].itertuples():
            if not healthy(r) or (r.player_id, k) in seen:
                continue
            before = allme[(allme.player_id == r.player_id) & (allme.week < k)]
            act = before[[healthy(x) for x in before.itertuples()]]
            starts = act[act.slot.isin(STARTER_SLOTS)]
            regular = r.slot in STARTER_SLOTS or (len(starts) >= 2 and len(starts) >= len(act) / 2)
            if not regular:
                continue
            bye = lambda w: w in BYEWK.get(r.nfl_team, set())
            now = allme[(allme.week == k) & (allme.player_id == r.player_id)]
            hurt_here = len(now) > 0 and not healthy(now.iloc[0]) and not bye(k)
            dropped_hurt = len(now) == 0 and r.player_id in on_ir_now and not any(w >= k for w in scored_wk.get(r.player_id, ()))
            if not (hurt_here or dropped_hurt):
                continue
            weeks, w = [], k
            while w <= WEEK and w not in scored_wk.get(r.player_id, ()):
                seen.add((r.player_id, w))
                if not bye(w):
                    st_w = allme[(allme.week == w) & allme.slot.isin(STARTER_SLOTS)]
                    ok = st_w[(st_w.position == r.position) | ((st_w.slot == "FLEX") & (r.position in ("RB", "WR", "TE")))]
                    if len(ok):
                        rep = ok.sort_values("points").iloc[0]; weeks.append((w, rep.player_name, rep.points))
                w += 1
            if weeks:
                INJ.append(dict(start=weeks[0][0], end=weeks[-1][0], name=r.player_name, pos=r.position,
                                avg=starts.points.mean(), weeks=weeks, dropped=bool(dropped_hurt)))
    return INJ

def team_data(T0):
    me = [r for r in rows if r["t"] == T0][0]
    allme = L26[L26.t == T0]
    mine = st_all[st_all.t == T0]
    tot_l = sum(LUCKCOL[T0].values()) or 1
    LUCK_SPLIT = {c: LUCKCOL[T0][c] / tot_l for c in LOSSC}
    LUCK_AMT = {c: LUCKCOL[T0][c] for c in LOSSC}

    started = set(zip(mine.nn, mine.week))
    OVR = [l for p in AD["players"] for l in p["log"]
           if l["week"] <= WEEK and (norm(p["name"]), l["week"]) in started and l["col"] in ("flag", "replay", "short") and l["pts"] >= 5.5]

    g = mine.groupby(["player_id", "player_name", "position", "nfl_team"]).agg(pts=("points", "sum"), n=("points", "size")).reset_index()
    CARRY = [(r.player_name, r.position, r.nfl_team if r.position != "DEF" else "", r.pts, r.n) for r in g.sort_values("pts", ascending=False).head(5).itertuples()]
    DRAG = [(r.player_name, r.position, r.nfl_team, r.n, r.pts / r.n) for r in g.itertuples()
            if r.position in POS_AVG and r.n >= MIN_STARTS and r.pts / r.n < POS_AVG[r.position]]
    DRAG = sorted(DRAG, key=lambda d: (d[4] - POS_AVG[d[1]]) * d[3])[:5]

    # Bench regrets: the bench players who most outscored a starter they could have replaced
    REG = []
    for k in range(1, WEEK + 1):
        w = allme[allme.week == k]; S = w[w.slot.isin(STARTER_SLOTS)]; B = w[w.slot == "BN"]
        for b in B.itertuples():
            for s in S.itertuples():
                if (s.slot == b.position or (s.slot == "FLEX" and b.position in ("RB", "WR", "TE"))) and b.points > s.points:
                    REG.append((k, b.player_name, b.position, b.points, s.player_name, s.points))
    seen = set()
    REGRETS = [x for x in sorted(REG, key=lambda x: -(x[3] - x[5])) if not ((x[0], x[1]) in seen or seen.add((x[0], x[1])))][:3]

    pp = mine[mine.position.isin(["QB", "RB", "WR", "TE"])]
    b = pp.sort_values("points", ascending=False).iloc[0]
    PERF_BEST = (b.player_name, b.position, b.nfl_team, int(b.week), b.points, f"projected {b.projected:.1f}")
    pp = pp.assign(d=pp.points - pp.projected); w_ = pp.sort_values("d").iloc[0]
    PERF_WORST = (w_.player_name, w_.position, w_.nfl_team, int(w_.week), w_.points, w_.projected, f"{w_.projected-w_.points:.1f} short of his projection")

    # Roster moves: the best player this team didn't draft, and how much it used the wire and trades
    mytx = TX[(TX.season == SEASON) & (TX.team_id == AB2ID[T0]) & ~TX.move_id.isin(UNDONE)]
    adds = mytx[mytx.action.isin(["FA ADDED", "WAIVER ADDED"])]
    drafted_ids = set(DR[(DR.season == SEASON) & (DR.team_id == AB2ID[T0])].player_id)
    nd = g[~g.player_id.isin(drafted_ids) & ~g.position.isin(["K", "DEF"])].sort_values("pts", ascending=False)
    BEST_ADD = None
    if len(nd):
        r = nd.iloc[0]
        acq = mytx[(mytx.player_id == r.player_id) & mytx.action.isin(["FA ADDED", "WAIVER ADDED", "TRADE_RECEIVED"])].sort_values("date")
        how = "Trade" if len(acq) and acq.iloc[0].action == "TRADE_RECEIVED" else "Waiver add"
        wk = int(allme[allme.player_id == r.player_id].week.min())
        BEST_ADD = (r.player_name, r.position, r.nfl_team, f"{how}, Week {wk}", int(r.n), r.pts, r.pts / r.n)
    rk_started = sorted(TEAMS, key=lambda t: -n_started[t]).index(T0) + 1
    add_pts = mine[mine.player_id.isin(set(adds.player_id))].points.sum()
    n_trades = mytx[mytx.action.str.startswith("TRADE")].move_id.nunique()
    lineups_by_week = [frozenset(allme[(allme.week == k) & allme.slot.isin(STARTER_SLOTS)].player_id) for k in range(1, WEEK + 1)]
    same = sum(1 for k in range(1, WEEK) if lineups_by_week[k] == lineups_by_week[k - 1])
    USAGE = [("Players started", str(n_started[T0]), f"{ordn(rk_started)} most in the league", "#13294B"),
             ("Waiver adds", str(adds.player_id.nunique()), f"Their starts scored {add_pts:.1f}", "#1F8A4C"),
             ("Trades", str(n_trades), "Completed this season", "#1F8A4C"),
             ("Same lineup", str(same), "Weeks with no changes", "#A5ACAF")]

    return dict(T0=T0, me=me, LUCK_SPLIT=LUCK_SPLIT, LUCK_AMT=LUCK_AMT, OVR_TD=len(OVR), OVR_PTS=sum(l["pts"] for l in OVR),
                CARRY=CARRY, DRAG=DRAG, INJ=injuries_for(T0, allme), PROJ=[PROJD[T0][k] for k in range(WEEK)],
                REGRETS=REGRETS, PERF_BEST=PERF_BEST, PERF_WORST=PERF_WORST, BEST_ADD=BEST_ADD, USAGE=USAGE,
                lost_total=sum(LUCKW[T0]))

TEAM_DATA = {t: team_data(t) for t in TEAMS}
print("Built", len(TEAM_DATA), "team pages' data")

## Draft & Moves

**Pick grades** use points over replacement at each position (QB12, RB30, WR30, TE12, K12, and DEF12), compared only with players drafted after the pick: two-thirds against the next 5 players at his position and one-third against everyone in the next round (a round and a half from Round 10 on). Anyone below replacement counts as replacement, since a player that bad can be swapped for one off waivers. The gap is in points per week, so the same scale holds all season.

**Injured picks** (out 5+ weeks, or on IR now and not playing) get a flag and can't be busts or a team's worst pick.

**Trades** are graded by best-possible lineup points from the trade on, with the trade against without it. **Pickups** are graded by points per start (2+ starts, no kickers or defenses).

In [ ]:
REPL_N = {"QB": 12, "RB": 30, "WR": 30, "TE": 12, "K": 12, "DEF": 12}

def grade_picks(picks, pool, nweeks, n_teams=12):
    repl = {}
    for pos, n in REPL_N.items():
        v = sorted(pool.get(pos) or [q["pts"] for q in picks if q["pos"] == pos], reverse=True)
        repl[pos] = v[min(n, len(v)) - 1] if v else 0.0
    for q in picks:
        q["vor"] = q["pts"] - repl.get(q["pos"], 0.0)
    win = lambda o: n_teams if o // n_teams < 9 else n_teams * 3 // 2
    for q in picks:
        after = [x for x in picks if x["o"] > q["o"]]
        same = [max(x["vor"], 0.0) for x in after if x["pos"] == q["pos"]][:5]; same += [0.0] * (5 - len(same))
        nxt = [max(x["vor"], 0.0) for x in after if x["o"] - q["o"] <= win(q["o"])]; nxt += [0.0] * (win(q["o"]) - len(nxt))
        q["exp_pos"] = sum(same) / len(same); q["exp_all"] = sum(nxt) / len(nxt)
        own = max(q["vor"], 0.0)
        q["over"] = (2 / 3) * (own - q["exp_pos"]) + (1 / 3) * (own - q["exp_all"])
        q["gap"] = q["over"] / nweeks
    return repl

def injured_players(Ls, week, current=False):
    # Out 5+ weeks (projected and scored 0 outside his bye, or dropped while out), or on IR now and not playing
    Ls = Ls[Ls.week <= week].drop_duplicates(["player_id", "week"])
    z = Ls[Ls.position != "DEF"].assign(zero=Ls.projected == 0).groupby(["nfl_team", "week"]).zero.mean().reset_index()
    bye = {t: set(g[g.zero >= 0.8].week) for t, g in z.groupby("nfl_team")}
    out = set()
    for pid, g in Ls[~Ls.position.isin(["K", "DEF"])].groupby("player_id"):
        g = g.sort_values("week")
        miss = [int(r.week) for r in g.itertuples() if r.projected == 0 and r.points == 0 and r.week not in bye.get(r.nfl_team, set())]
        last = g.iloc[-1]; n = len(miss)
        if int(last.week) in miss and last.week < week:
            n += week - int(last.week)
        on_ir = current and last.injury == "INJURY_RESERVE" and (int(last.week) in miss or last.week < week)
        if n >= 5 or on_ir:
            out.add(g.player_name.iloc[0])
    return out

def ranks_by_position(picks):
    for pos in set(q["pos"] for q in picks):
        pp = [q for q in picks if q["pos"] == pos]
        for i, q in enumerate(sorted(pp, key=lambda q: q["o"])): q["drank"] = f"{pos}{i+1}"
        for i, q in enumerate(sorted(pp, key=lambda q: -q["pts"])): q["prank"] = f"{pos}{i+1}"

d26 = DR[DR.season == SEASON].sort_values("overall")
ROUNDS = int(d26["round"].max())
ORDER = [ID2AB[x] for x in d26[d26["round"] == 1].sort_values("round_pick").team_id]
LAST_TEAM = L26.sort_values("week").groupby("player_id").nfl_team.last()
HURT = injured_players(L26, WEEK, current=True)
_pp = L26.drop_duplicates(["player_id", "week"]).groupby(["player_id", "position"]).points.sum().reset_index()
ALLPTS = {p: list(g.points) for p, g in _pp.groupby("position")}

NFL, picks = {}, []
for r in d26.itertuples():
    tm_ = LAST_TEAM.get(r.player_id, "")
    NFL[r.player_name] = "" if r.position == "DEF" else (tm_ if isinstance(tm_, str) and tm_ else "FA")
    picks.append(dict(o=int(r.overall) - 1, r=int(r.round), p=int(r.round_pick), t=ID2AB[r.team_id], name=r.player_name,
                      pos=r.position, pts=float(PTS.get(r.player_id, 0.0)), pid=r.player_id))
REPL = grade_picks(picks, ALLPTS, WEEK, N)
ranks_by_position(picks)
for q in picks:
    q["inj"] = q["name"] in HURT

# Trades, both sides, in best possible lineup points with and without the trade
ROST = {(t, k): L26[(L26.t == t) & (L26.week == k)] for t in TEAMS for k in range(1, WEEK + 1)}
WEEKPTS = {(r.player_id, r.week): (r.points, r.position) for r in L26.itertuples()}
def best_lineup(players):
    df = pd.DataFrame([dict(player_id=p, points=pts, position=pos, slot="BN") for p, (pts, pos) in players.items()])
    return optimal_df(df)[0] if len(df) else 0.0
tr = TX[(TX.season == SEASON) & TX.action.str.startswith("TRADE") & ~TX.move_id.isin(UNDONE)]
TRADES = []
for mid, g in tr.groupby("move_id"):
    a_id, b_id = sorted(set(g.team_id))[:2]; A, B = ID2AB[a_id], ID2AB[b_id]
    recvA = g[(g.team_id == a_id) & (g.action == "TRADE_RECEIVED")]; recvB = g[(g.team_id == b_id) & (g.action == "TRADE_RECEIVED")]
    got = lambda t, ids: [k for k in range(1, WEEK + 1) if set(ids) & set(ROST[(t, k)].player_id)]
    wk = min((got(A, recvA.player_id) + got(B, recvB.player_id)) or [WEEK])
    def side(t, recv, sent):
        w = wo = 0.0
        for k in range(wk, WEEK + 1):
            roster = ROST[(t, k)]; roster = roster[roster.slot != "IR"]
            have = {r.player_id: (r.points, r.position) for r in roster.itertuples()}
            w += best_lineup(have)
            alt = {p: v for p, v in have.items() if p not in set(recv)}
            for p in sent:
                if (p, k) in WEEKPTS: alt[p] = WEEKPTS[(p, k)]
            wo += best_lineup(alt)
        return w, wo
    aw, awo = side(A, list(recvA.player_id), list(recvB.player_id)); bw, bwo = side(B, list(recvB.player_id), list(recvA.player_id))
    for nm, pid in zip(g.player_name, g.player_id):
        NFL.setdefault(nm, LAST_TEAM.get(pid, ""))
    TRADES.append(dict(wk=wk, a=A, ra=[(r.player_name, r.position) for r in recvA.itertuples()], a_with=aw, a_without=awo,
                       b=B, rb=[(r.player_name, r.position) for r in recvB.itertuples()], b_with=bw, b_without=bwo))
TRADES.sort(key=lambda d: d["wk"])

# Pickups: points per start for the team that added him
ad = TX[(TX.season == SEASON) & TX.action.isin(["FA ADDED", "WAIVER ADDED"]) & ~TX.move_id.isin(UNDONE)].sort_values("date")
ADDS, seen = [], set()
for r in ad.itertuples():
    t = ID2AB[r.team_id]
    if (t, r.player_id) in seen:
        continue
    seen.add((t, r.player_id))
    s_ = L26[(L26.t == t) & (L26.player_id == r.player_id) & L26.slot.isin(STARTER_SLOTS)]
    if len(s_) >= 2 and r.position not in ("K", "DEF"):
        first = int(L26[(L26.t == t) & (L26.player_id == r.player_id)].week.min())
        ADDS.append((r.player_name, r.position, t, first, s_.points.sum(), len(s_)))
        NFL[r.player_name] = LAST_TEAM.get(r.player_id, "")
ADDS = sorted(ADDS, key=lambda a: -a[4] / a[5])[:8]
mv = TX[(TX.season == SEASON) & ~TX.move_id.isin(UNDONE)]
MOVES = {t: mv[mv.team_id == AB2ID[t]].move_id.nunique() for t in TEAMS}
print(len(picks), "picks ·", len(HURT), "injured ·", len(TRADES), "trades ·", len(ADDS), "pickups")

## History

Every finished season: weekly scores, head-to-head games (regular season and winners bracket), standings, final places, and the playoff bracket. Head to head and rivalries also count this season's games so far.

In [ ]:
SEASONS = sorted(int(y) for y in TM.season.unique() if y < SEASON)
HIST = {}
for y in SEASONS:
    s = SCH[SCH.season == y]; reg = s[s.bracket == "NONE"]; Gy = int(reg.week.max())
    sc = {t: [0.0] * Gy for t in TEAMS}; hh = {}
    for r in reg.itertuples():
        a, b = ID2AB[r.home_team_id], ID2AB[r.away_team_id]
        sc[a][r.week - 1] = r.home_score; sc[b][r.week - 1] = r.away_score
    for r in s[s.bracket.isin(["NONE", "WINNERS_BRACKET"])].itertuples():
        a, b = ID2AB[r.home_team_id], ID2AB[r.away_team_id]
        hh.setdefault((a, b), []).append((y, r.week - 1, r.home_score, r.away_score))
    tt = TM[TM.season == y]
    stand = [ID2AB[r.team_id] for r in tt.sort_values("seed").itertuples()]
    fin = [ID2AB[r.team_id] for r in tt.sort_values("final_place").itertuples()]
    wb = s[s.bracket == "WINNERS_BRACKET"].sort_values("week")
    po = sorted(set(ID2AB[x] for x in wb[wb.week == wb.week.min()][["home_team_id", "away_team_id"]].values.ravel()), key=stand.index)
    HIST[y] = dict(sc=sc, w={ID2AB[r.team_id]: int(r.wins) for r in tt.itertuples()}, G=Gy, h2=hh, stand=stand,
                   champ=fin[0], runner=fin[1], po=po, last=fin[-1], fin=fin, wb=wb, final_week=int(wb.week.max()))
FINAL = {y: HIST[y]["fin"] for y in SEASONS}
CUR = {r["t"]: (r["w"], r["l"], sum(SC[r["t"]])) for r in rows}      # this season so far, for the all-time table

# Playoff results by team: W and L, plus CW and CL for winning and losing the final
PO = {t: [] for t in TEAMS}
for y in SEASONS:
    H = HIST[y]
    for r in H["wb"].itertuples():
        a, b = ID2AB[r.home_team_id], ID2AB[r.away_team_id]; aw = r.home_score > r.away_score
        fin = r.week == H["final_week"]
        PO[a].append((y, ("CW" if aw else "CL") if fin else ("W" if aw else "L")))
        PO[b].append((y, ("CL" if aw else "CW") if fin else ("L" if aw else "W")))

# Head-to-head games, this season included
H2ALL = {y: HIST[y]["h2"] for y in SEASONS}
H2ALL[SEASON] = {}
for t in TEAMS:
    for k in range(WEEK):
        o = opp[t][k]
        if t < o:
            H2ALL[SEASON].setdefault((t, o), []).append((SEASON, k, SC[t][k], SC[o][k]))

def build_rivals():
    # The three most-played pairs, closest records first among ties
    pairs = {}
    for y in SEASONS + [SEASON]:
        for (a, b), gs in H2ALL[y].items():
            key = tuple(sorted((a, b)))
            for (_, k, sa, sb) in gs:
                wa = (sa > sb) if key[0] == a else (sb > sa)
                pairs.setdefault(key, []).append((y, k, wa, abs(sa - sb)))
    ranked = sorted(pairs.items(), key=lambda kv: (-len(kv[1]), abs(sum(g[2] for g in kv[1]) * 2 - len(kv[1]))))[:3]
    out = []
    for (a, b), gs in ranked:
        wa = sum(1 for g in gs if g[2]); last = max(gs, key=lambda g: (g[0], g[1]))
        out.append((a, b, len(gs), wa, len(gs) - wa, f"Average margin {sum(g[3] for g in gs)/len(gs):.1f} · last met Week {last[1]+1}, {last[0]}"))
    return out
RIVALS = build_rivals()

# Longest win streak, regular season only, across every season
STREAK = (0, None, None, None)
for t in TEAMS:
    seq = []
    for y in SEASONS:
        oppk = {}
        for (a, b), gs in HIST[y]["h2"].items():
            for (_, k, sa, sb) in gs:
                if k < HIST[y]["G"]:
                    if a == t: oppk[k] = sa > sb
                    if b == t: oppk[k] = sb > sa
        seq += [(y, oppk[k]) for k in sorted(oppk)]
    seq += [(SEASON, SC[t][k] > SC[opp[t][k]][k]) for k in range(WEEK)]
    run, start = 0, None
    for y, wn in seq:
        if wn:
            run += 1; start = start or y
            if run > STREAK[0]: STREAK = (run, t, start, y)
        else:
            run, start = 0, None
print("Seasons:", SEASONS, "· longest streak:", STREAK)

## Season superlatives

Eight for every finished season: MVP (most points as a starter), steal and bust of the draft (graded the same way as Draft & Moves, injured players can't be the bust), best pickup (most points from an undrafted starter), highest score, most points, toughest schedule (most points against), and best lineups (highest lineup score).

In [ ]:
def season_super(y):
    Ly = LU[(LU.season == y) & (LU.week <= HIST[y]["G"])].copy(); Ly["t"] = Ly.team_id.map(ID2AB)
    reg_weeks = HIST[y]["G"]
    st = Ly[Ly.slot.isin(STARTER_SLOTS)]
    out = []
    g = st.groupby(["player_id", "player_name", "position", "nfl_team", "t"]).points.sum().reset_index().sort_values("points", ascending=False)
    m = g.iloc[0]; out.append(("MVP", m.player_name, f"{m.position} · {m.nfl_team}", f"[Team {m.t}]", f"{m.points:.1f} points as a starter"))
    pts = Ly.drop_duplicates(["player_id", "week"]).groupby("player_id").points.sum()
    dy = DR[DR.season == y].sort_values("overall")
    pk = [dict(o=int(r.overall) - 1, r=int(r.round), p=int(r.round_pick), t=ID2AB[r.team_id], name=r.player_name, pos=r.position,
               pts=float(pts.get(r.player_id, 0))) for r in dy.itertuples()]
    pool = Ly.drop_duplicates(["player_id", "week"]).groupby(["player_id", "position"]).points.sum().reset_index()
    grade_picks(pk, {p: list(gg.points) for p, gg in pool.groupby("position")}, reg_weeks)
    ranks_by_position(pk)
    pl = [q for q in pk if q["pos"] not in ("K", "DEF")]
    hurt = injured_players(Ly, reg_weeks)
    s_ = max(pl, key=lambda q: q["gap"]); b_ = min([q for q in pl if q["r"] <= 7 and q["name"] not in hurt], key=lambda q: q["gap"])
    nfl = lambda nm: Ly[Ly.player_name == nm].nfl_team.iloc[-1] if (Ly.player_name == nm).any() else ""
    out.append(("Steal of the draft", s_["name"], f"{s_['pos']} · {nfl(s_['name'])}", f"[Team {s_['t']}]", f"{s_['r']}.{s_['p']:02d} · drafted as {s_['drank']}, performing as {s_['prank']}"))
    out.append(("Biggest bust", b_["name"], f"{b_['pos']} · {nfl(b_['name'])}", f"[Team {b_['t']}]", f"{b_['r']}.{b_['p']:02d} · drafted as {b_['drank']}, performing as {b_['prank']}"))
    und = st[~st.player_id.isin(set(dy.player_id)) & ~st.position.isin(["K", "DEF"])]
    u = und.groupby(["player_name", "position", "nfl_team", "t"]).points.agg(["sum", "size"]).reset_index().sort_values("sum", ascending=False)
    if len(u):
        x = u.iloc[0]; out.append(("Best pickup", x.player_name, f"{x.position} · {x.nfl_team}", f"[Team {x.t}]", f"{x['sum']:.1f} in {x['size']} starts"))
    sc = HIST[y]["sc"]
    hi = max(((t, k, sc[t][k]) for t in TEAMS for k in range(len(sc[t]))), key=lambda z: z[2])
    out.append(("Highest score", f"{hi[2]:.1f}", f"Week {hi[1]+1}", f"[Team {hi[0]}]", "Best single week of the season"))
    tt = TM[TM.season == y]; pfm = tt.sort_values("points_for", ascending=False).iloc[0]; pam = tt.sort_values("points_against", ascending=False).iloc[0]
    out.append(("Most points", f"[Team {ID2AB[pfm.team_id]}]", "", f"[Team {ID2AB[pfm.team_id]}]", f"{pfm.points_for:,.1f} points for"))
    out.append(("Toughest schedule", f"[Team {ID2AB[pam.team_id]}]", "", f"[Team {ID2AB[pam.team_id]}]", f"{pam.points_against:,.1f} points against"))
    eff = {}
    for (t, k), gg in Ly.groupby(["t", "week"]):
        o, _ = optimal_df(gg); eff.setdefault(t, [0, 0])
        eff[t][0] += gg[gg.slot.isin(STARTER_SLOTS)].points.sum(); eff[t][1] += o
    be = max(eff, key=lambda t: eff[t][0] / eff[t][1])
    out.append(("Best lineups", f"[Team {be}]", "", f"[Team {be}]", f"{100*eff[be][0]/eff[be][1]:.1f}% lineup score"))
    return out

SUPER = {y: season_super(y) for y in SEASONS}
print({y: SUPER[y][0][1] for y in SEASONS})

## League home

The home page's matchups, top six in the standings, the ticker, and the schedule-luck list.

In [ ]:
M = [(f"[Team {a}]", sa, f"[Team {b}]", sb) for a, sa, b, sb in GAMES]
ST = [(f"[Team {r['t']}]", f"{r['w']}-{r['l']}", f"{r['ap_w']}-{r['ap_l']}", r["pf"]) for r in rows[:6]]
tl = {t: sum(LUCKW[t]) for t in TEAMS}
unl = max(tl, key=tl.get); lkst = min(tl, key=tl.get)
bestap = max(rows, key=lambda r: r["ap_w"])
TICK = [("Top score", f"[Team {HI10}]", f"{SC10D[HI10]:.1f} points"),
        ("Closest game", f"[Team {CLOSE10[0]}] over [Team {CLOSE10[2]}]", f"{CLOSE10[1]:.1f} to {CLOSE10[3]:.1f}"),
        ("Unluckiest team", f"[Team {unl}]", f"{tl[unl]:.1f} points lost this season"),
        ("Biggest blowout", f"[Team {BLOW10[0]}] over [Team {BLOW10[2]}]", f"won by {BLOW10[1]-BLOW10[3]:.1f}"),
        ("Most points in a loss", f"[Team {ML10[2]}]", f"{ML10[3]:.1f}"),
        ("Best all-play record", f"[Team {bestap['t']}]", f"{bestap['ap_w']}-{bestap['ap_l']}")]
sl = sorted(rows, key=lambda r: -r["luck"]); SL = [sl[0], sl[1], sl[-1]]

## Save

Everything `02_build_site` needs goes into one file. League-wide numbers are saved once; the team and matchup numbers are saved per team and per game.

In [ ]:
KEEP = ["LEAGUE_ID", "LEAGUE_NAME", "SCORING", "SEASON", "WEEK", "WEEKS", "PLAYOFF_TEAMS", "REG_WEEKS", "TEAMS", "N", "NAME", "MGR", "ID2AB", "AB2ID",
        "SC", "SCORES", "opp", "OPT", "BENCH", "PROJD", "LINEW", "MAXPF", "LUCKW", "LUCKCOL", "LOSSC", "POS_SPLIT", "POS_AVG", "MIN_STARTS",
        "GAMES", "GAMES10", "TOP3", "SC10D", "HI10", "CLOSE10", "BLOW10", "ML10", "TILES", "AWARDS", "TOP", "WEEK_SCORES", "MEDIAN",
        "rows", "SWAP", "LEFT", "ROUNDS", "ORDER", "NFL", "picks", "REPL", "HURT", "TRADES", "ADDS", "MOVES",
        "SEASONS", "HIST", "FINAL", "CUR", "PO", "H2ALL", "RIVALS", "STREAK", "SUPER", "M", "ST", "TICK", "SL", "tl", "unl", "lkst"]
WEEKS = WEEK
league = {k: globals()[k] for k in KEEP}
league["LUCKCOL"] = {t: dict(v) for t, v in LUCKCOL.items()}
pd.to_pickle(dict(league=league, teams=TEAM_DATA, matchups=MATCHUPS, luck=AD), OUT_FILE)
print("Saved", OUT_FILE, f"({OUT_FILE.stat().st_size/1e6:.1f} MB)")